# ML-00 real-data smoke test

실제 ML feature parquet와 feature catalog를 사용해 `ml_io`, `ml_train`, `ml_val`, `ml_test`의 학습/검증/최종 테스트 흐름을 점검합니다.

In [ ]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

In [ ]:
# ============================================================
# 0. Local ml_utils.py and module path
# ============================================================
# ml_utils.py와 모듈 파일은 노트북과 같은 폴더에 있다고 가정
# 다르면 사용자가 직접 절대경로로 지정
MODULE_DIR: str | Path = Path.cwd()
MODULE_DIR = Path(MODULE_DIR).expanduser()
MODULE_DIR = MODULE_DIR.resolve()
# ml-00처럼 하이픈 있는 폴더는 일반 패키지 import 어렵기 때문에 경로만 추가
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

# 팀유틸 import 및 시드 고정
import ml_utils
SEED = 42
ml_utils.set_seed(SEED, use_torch=False)

BASE_DIR = ml_utils.BASE_DIR
DATA_DIR = ml_utils.DATA_DIR
RAW_DIR = ml_utils.RAW_DIR
PROCESSED_DIR = ml_utils.PROCESSED_DIR
PROJECT_ROOT = BASE_DIR
ML_UTILS_PATH = Path(ml_utils.__file__).resolve()

# ml 파트 모듈 import
import ml_io
import ml_test
import ml_train
import ml_val

# 노트북에서 모듈 코드를 수정한 뒤 커널 재시작 없이 다시 반영하기 위한 reload
# 일반 script 실행에서는 없어도 되지만, 노트북 실험 편의를 위해 메모
# importlib.reload(ml_io)
# importlib.reload(ml_train)
# importlib.reload(ml_val)
# importlib.reload(ml_test)

In [ ]:

# ============================================================
# 1. Run settings
# ============================================================
# 데이터셋을 바꾸려면 EXPERIMENT_NAME, DATA_DIR, 파일명, SAMPLE_ROWS만 이 설정 셀에서 수정
EXPERIMENT_NAME = "ml-00-smoketest-rerun"

# RUN_SMOKETEST_CASE_CHECKS=True이면 XGBoost 학습 없이 ml_io 레벨에서 fixture 검증만 실행
#   - smoketest fixture 데이터와 *_smoketest feature catalog를 쓸 때만 True 권장
#   - 실제 데이터 경로를 입력한 상태에서는 False로 두어 fixture 전용 bad/contract case와 섞이지 않게 함
#   - contract_cases: 정상 통과해야 하는 운영 상황,  bad_cases: 명확한 에러로 실패해야 하는 상황
RUN_SMOKETEST_CASE_CHECKS = False

# ---------------------------------------------------
# sample/full 실행 설정
# - smoketest fixture는 작고 고정된 데이터라 보통 None으로 전체 사용
# - 실제 데이터는 100_000 같은 숫자를 두면 각 split에서 일부 행만 읽어 빠르게 점검
# - 최종 학습/검증은 None으로 두어 전체 split 사용
# ---------------------------------------------------
SAMPLE_ROWS: int | None = None
RUN_TRAIN_AND_VALIDATION = False

# final test는 모델/feature/threshold 선택이 끝난 뒤 확인용으로만 실행
# 기본값 False를 권장, True로 바꾸면 기존 metrics_test.json이 있을 때 OVERWRITE_POLICY['test'] 설정에 따라 차단
RUN_FINAL_TEST = False


# ============================================================
# 2. Input/output settings
# ============================================================
# ---------------------------------------------------
# 입력 데이터 경로와 파일명 직접 입력
# ---------------------------------------------------
# 이 블록은 데이터 입력의 source of truth
# smoketest fixture를 쓸 때는 아래 기본값 그대로 사용
# 실제 데이터로 바꿀 때는 DATA_DIR과 파일명 4개만 직접 수정
# 예: DATA_DIR = BASE_DIR / "ml" / "ml-00_baseline" / "inputs" / "ml_features" / "ml_exp00" / "run_001"
# 예: TRAIN_FILE_NAME = "ml_exp00_Xy_train.parquet"
DATA_DIR: str | Path = BASE_DIR / "ml" / "ml-96_smoketest" / "ml_features"

TRAIN_FILE_NAME = "ml_exp00_Xy_train_smoketest.parquet"
VAL_FILE_NAME = "ml_exp00_Xy_val_smoketest.parquet"
TEST_FILE_NAME = "ml_exp00_Xy_test_smoketest.parquet"
FEATURE_COLUMNS_FILE_NAME = "ml_feature_columns_smoketest.csv"

RUN_KIND = f"sample_{SAMPLE_ROWS}" if SAMPLE_ROWS is not None else "full" # 아웃풋 디렉터리 이름에 sample_1000 같이 샘플 여부 표시을 위해 RUN_KIND 설정

OUT_DIR: str | Path = BASE_DIR / "ml" / "ml-00_baseline" / "outputs" / "train_val_test" / EXPERIMENT_NAME / RUN_KIND
OUT_DIR = ml_io.resolve_project_path(OUT_DIR, project_root=PROJECT_ROOT)

# ---------------------------------------------------
# 입력 경로 객체 단일화
# ---------------------------------------------------
# INPUT_PATHS는 train/val/test/feature catalog 경로를 한 번만 조립한 객체
# 이후 preview, smoketest case check, train, validation, final test는 이 객체만 재사용
INPUT_PATHS = ml_io.make_input_paths(
    data_dir=DATA_DIR,
    train_file_name=TRAIN_FILE_NAME,
    val_file_name=VAL_FILE_NAME,
    test_file_name=TEST_FILE_NAME,
    feature_columns_file_name=FEATURE_COLUMNS_FILE_NAME,
    project_root=PROJECT_ROOT,
)

# ============================================================
# 3. Execution policy and model settings
# ============================================================
# 아래 설정들은 노트북 실행 기준의 source of truth
# 모듈 파일에 기본값이 있지만, 필요시 노트북에서 config 객체를 만들 때 아래 값을 넘겨 기본값을 덮어씀
LABEL_COL = "label"

# overwrite 정책
OVERWRITE_POLICY = {
    "train": False,
    "val": False,
    "test": False,
}

# XGBoost 학습 파라미터
# 이 dict는 ml_train.XGBTrainConfig의 기본값을 노트북에서 한 번에 덮어쓰기 위한 설정
# 값을 바꾸려면 run_train_and_validation() 내부가 아니라 여기만 수정
# 모듈 기본값을 쓰고 싶다면 run_train_and_validation() 안의 `**XGB_PARAMS,` 줄 주석 처리
XGB_PARAMS = {
    "n_estimators": 50,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "min_child_weight": 1.0,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "gamma": 0.0,
    "early_stopping_rounds": 10,
    "n_jobs": 1,
}

# threshold 선택 정책
# 기본은 validation F1이 최대가 되는 threshold를 자동 선택하는 max_f1
# 운영/비교 목적상 threshold를 직접 고정해야 하면 THRESHOLD_STRATEGY='manual', MANUAL_THRESHOLD=0.42처럼 설정
THRESHOLD_STRATEGY = "max_f1"
MANUAL_THRESHOLD = None

# 경로 확인 출력
print("="*70)
print("PROJECT_ROOT   :", PROJECT_ROOT)
print("ml_utils.py    :", ML_UTILS_PATH)
print("MODULE_DIR     :", MODULE_DIR)
print("="*70)
print("BASE_DIR       :", BASE_DIR)
print("DATA_DIR       :", DATA_DIR)
print("RAW_DIR        :", RAW_DIR)
print("PROCESSED_DIR  :", PROCESSED_DIR)
print("OUT_DIR        :", OUT_DIR)
print("SMOKE_CASES    :", RUN_SMOKETEST_CASE_CHECKS)
print("="*70)
print("TRAIN_FILE     :", TRAIN_FILE_NAME)
print("VAL_FILE       :", VAL_FILE_NAME)
print("TEST_FILE      :", TEST_FILE_NAME)
print("FEATURE_COLS   :", FEATURE_COLUMNS_FILE_NAME)
print("="*70)


In [ ]:
# 설정값은 위의 1~3번 설정 셀로 통합됨
# 설정을 바꿀 때는 위 설정 셀의 OVERWRITE_POLICY / XGB_PARAMS / THRESHOLD_STRATEGY / MANUAL_THRESHOLD만 수정

In [ ]:
# ============================================================
# 학습전 입력 데이터와 feature catalog를 미리 확인
# ============================================================
def preview_inputs(paths: ml_io.InputPaths) -> list[str]:
    """학습 전에 입력 경로와 feature 목록을 출력해 설정이 올바른지 확인"""
    ml_io.print_input_paths(paths)    # train/validation/test/feature catalog 경로를 화면에 출력
    # 필수 입력 파일 존재 여부 확인, RUN_FINAL_TEST=True이면 test 파일도 확인
    ml_io.require_input_files(paths, require_test=RUN_FINAL_TEST)
    feature_columns = ml_io.load_feature_columns(
        paths.feature_columns_path,
        label_col=LABEL_COL,    # feature catalog에 정답 label 컬럼이 포함됐는지 검사하기 위한 기준 컬럼명 (확인용)
    )
    print("feature_count:", len(feature_columns))                               # 사용할 feature 개수
    print("feature_columns_hash:", ml_io.feature_columns_hash(feature_columns)) # feature 목록의 hash 출력
    print("first_features:", feature_columns[:10])
    print("feature_columns:", feature_columns)
    print("out_dir:", OUT_DIR)

    return feature_columns # 이후 단계에서 재사용할 수 있도록 feature 컬럼 목록을 반환

In [ ]:
# ============================================================
# Optional. Smoketest fixture contract/bad-case checks
# ============================================================
def expect_error(name, fn, expected_message: str) -> None:
    """bad case가 기대한 에러 메시지로 실패하는지 확인합니다."""
    try:
        fn()
    except Exception as exc:
        message = str(exc)
        if expected_message not in message:
            raise AssertionError(
                f"{name}: expected message containing {expected_message!r}, got {message!r}"
            ) from exc
        print(f"[EXPECTED ERROR] {name}: {type(exc).__name__}: {message.splitlines()[0]}")
    else:
        raise AssertionError(f"{name}: expected failure but succeeded.")

def run_smoketest_case_checks() -> None:
    """ml-96_smoketest fixture의 정상/오류 검출 케이스를 노트북에서 확인합니다."""
    # RUN_SMOKETEST_CASE_CHECKS는 fixture 전용 검증 실행 여부
    if not RUN_SMOKETEST_CASE_CHECKS:
        print("Smoketest case checks skipped because RUN_SMOKETEST_CASE_CHECKS=False.")
        return

    smoke_dir = BASE_DIR / "ml" / "ml-96_smoketest"
    expected_smoke_data_dir = smoke_dir / "ml_features"
    if Path(DATA_DIR).resolve() != expected_smoke_data_dir.resolve():
        raise ValueError(
            "RUN_SMOKETEST_CASE_CHECKS=True requires DATA_DIR to point to the smoketest fixture ml_features directory. "
            f"DATA_DIR={Path(DATA_DIR).resolve()}, expected={expected_smoke_data_dir.resolve()}"
        )
    contract_dir = smoke_dir / "contract_cases"
    bad_dir = smoke_dir / "bad_cases"

    # 입력 경로는 설정 셀에서 만든 INPUT_PATHS를 재사용
    # 여기서 다시 make_input_paths()를 호출하면 설정 셀과 함수 내부 경로가 어긋날 수 있으므로 단일 객체만 사용
    paths = INPUT_PATHS
    feature_columns = ml_io.load_feature_columns(paths.feature_columns_path, label_col=LABEL_COL)

    # contract case는 통과해야 정상, parquet 컬럼 순서 변경/추가 컬럼을 안전하게 처리하는지 확인
    contract_cases = {
        "shuffled_columns": contract_dir / "val_shuffled_columns_smoketest_contract_cases.parquet",
        "extra_column": contract_dir / "val_extra_column_smoketest_contract_cases.parquet",
    }
    for name, path in contract_cases.items():
        x, y = ml_io.load_split(path, feature_columns, label_col=LABEL_COL, expected_split="val")
        if list(x.columns) != feature_columns:
            raise AssertionError(f"{name}: feature order mismatch.")
        print(f"[CONTRACT PASS] {name}: x_shape={x.shape}, y_rows={len(y)}")

    # bad case는 조용히 통과하면 안 됨 기대한 에러로 실패해야 정상
    expect_error(
        "missing_feature",
        lambda: ml_io.load_split(
            bad_dir / "val_missing_feature_smoketest_bad_cases.parquet",
            feature_columns=feature_columns,
            label_col=LABEL_COL,
            expected_split="val",
        ),
        "missing required columns",
    )
    expect_error(
        "missing_label",
        lambda: ml_io.load_split(
            bad_dir / "val_missing_label_smoketest_bad_cases.parquet",
            feature_columns=feature_columns,
            label_col=LABEL_COL,
            expected_split="val",
        ),
        "missing required columns",
    )
    expect_error(
        "wrong_split",
        lambda: ml_io.load_split(
            bad_dir / "val_wrong_split_smoketest_bad_cases.parquet",
            feature_columns=feature_columns,
            label_col=LABEL_COL,
            expected_split="val",
        ),
        "Unexpected split values",
    )
    expect_error(
        "nan_feature",
        lambda: ml_io.load_split(
            bad_dir / "val_nan_feature_smoketest_bad_cases.parquet",
            feature_columns=feature_columns,
            label_col=LABEL_COL,
            expected_split="val",
        ),
        "NaN values",
    )
    expect_error(
        "wrong_dtype",
        lambda: ml_io.load_split(
            bad_dir / "val_wrong_dtype_smoketest_bad_cases.parquet",
            feature_columns=feature_columns,
            label_col=LABEL_COL,
            expected_split="val",
        ),
        "All features must be numeric",
    )
    expect_error(
        "null_explosion",
        lambda: ml_io.load_split(
            bad_dir / "val_null_explosion_smoketest_bad_cases.parquet",
            feature_columns=feature_columns,
            label_col=LABEL_COL,
            expected_split="val",
        ),
        "NaN values",
    )
    expect_error(
        "inf_feature",
        lambda: ml_io.load_split(
            bad_dir / "val_inf_feature_smoketest_bad_cases.parquet",
            feature_columns=feature_columns,
            label_col=LABEL_COL,
            expected_split="val",
        ),
        "infinite values",
    )
    expect_error(
        "single_class",
        lambda: ml_io.load_split(
            bad_dir / "val_single_class_smoketest_bad_cases.parquet",
            feature_columns=feature_columns,
            label_col=LABEL_COL,
            expected_split="val",
        ),
        "Both classes are required",
    )
    expect_error(
        "label_leak_catalog",
        lambda: ml_io.load_feature_columns(
            bad_dir / "val_label_leak_catalog_smoketest_bad_cases.csv",
            label_col=LABEL_COL,
        ),
        "Data leakage risk",
    )
    expect_error(
        "duplicate_feature_catalog",
        lambda: ml_io.load_feature_columns(
            bad_dir / "val_duplicate_feature_catalog_smoketest_bad_cases.csv",
            label_col=LABEL_COL,
        ),
        "Duplicated selected feature columns",
    )
    expect_error(
        "empty_feature_catalog",
        lambda: ml_io.load_feature_columns(
            bad_dir / "val_empty_feature_catalog_smoketest_bad_cases.csv",
            label_col=LABEL_COL,
        ),
        "No usable feature columns",
    )
    expect_error(
        "invalid_used_in_ml_catalog",
        lambda: ml_io.load_feature_columns(
            bad_dir / "val_invalid_used_in_ml_catalog_smoketest_bad_cases.csv",
            label_col=LABEL_COL,
        ),
        "unsupported values",
    )
    expect_error(
        "forbidden_name_catalog",
        lambda: ml_io.load_feature_columns(
            bad_dir / "val_forbidden_name_catalog_smoketest_bad_cases.csv",
            label_col=LABEL_COL,
        ),
        "Data leakage risk",
    )
    print("[SMOKETEST CASE CHECKS PASS]")


if RUN_SMOKETEST_CASE_CHECKS:
    run_smoketest_case_checks()

In [ ]:
def run_train_and_validation(paths: ml_io.InputPaths) -> tuple[ml_train.XGBTrainResult, ml_val.ValidationResult]:
    """train 학습과 validation threshold 선택을 순서대로 실행합니다."""
    # XGBoost 학습 설정
    # seed, overwrite, XGBoost hyperparameter는 위 설정 셀의 값을 사용
    # XGB_PARAMS를 **로 넘기기 때문에 노트북 설정이 ml_train.XGBTrainConfig의 기본값을 덮어씀
    # 모듈 기본값을 사용하려면 아래 train_config에서 `**XGB_PARAMS,` 줄을 주석 처리
    train_config = ml_train.XGBTrainConfig(
        train_path=paths.train_path,
        val_path=paths.val_path,
        feature_columns_path=paths.feature_columns_path,
        output_dir=OUT_DIR,
        project_root=PROJECT_ROOT,

        label_col=LABEL_COL,
        sample_rows=SAMPLE_ROWS,
        overwrite=OVERWRITE_POLICY["train"],
        allow_nan=False,
        seed=SEED,
        **XGB_PARAMS,
    )
    train_result = ml_train.train_xgb(train_config)

    # 학습 결과 핵심 artifact 경로를 출력
    print("train_summary_path:", train_result.train_summary_path)
    print("model_path:", train_result.model_path)
    print("train_rows:", train_result.train_rows)
    print("val_rows_for_early_stopping:", train_result.val_rows)
    print("scale_pos_weight:", train_result.scale_pos_weight)

    # validation split에서 threshold를 선택
    # 여기서는 학습 때 저장된 model.pkl과 feature_columns.json을 사용
    # threshold_strategy와 manual_threshold도 위 설정 셀 값을 그대로 전달
    # 기본값 max_f1이면 validation F1 최대 threshold를 자동 선택하고, manual이면 사람이 지정한 threshold를 사용
    # test 단계에서는 여기서 저장된 threshold.json만 사용해야 하며 test set에서 threshold를 다시 고르면 데이터 누수 위험이 있음
    val_config = ml_val.ValidationConfig(
        val_path=paths.val_path,
        output_dir=OUT_DIR,
        project_root=PROJECT_ROOT,
        label_col=LABEL_COL,
        sample_rows=SAMPLE_ROWS,
        allow_nan=False,
        overwrite=OVERWRITE_POLICY["val"],
        threshold_strategy=THRESHOLD_STRATEGY,
        manual_threshold=MANUAL_THRESHOLD,
    )
    val_result = ml_val.validate_xgb(val_config)

    print("threshold_path:", val_result.threshold_path)
    print("val_metrics:", val_result.val_metrics["summary"])

    return train_result, val_result


def run_final_test(paths: ml_io.InputPaths) -> ml_test.TestResult:
    """full train/validation이 끝난 뒤 최종 test split 평가를 실행합니다."""
    # final test는 sample 실행 결과로 수행하면 안됨.
    # SAMPLE_ROWS가 None이 아니면 train/validation도 일부 행만 사용한 상태이므로 test 평가 차단
    if SAMPLE_ROWS is not None:
        raise ValueError(
            "Final test is blocked in this example because SAMPLE_ROWS is not None. "
            "Set SAMPLE_ROWS=None, rerun train/validation, then set RUN_FINAL_TEST=True."
        )
    if paths.test_path is None:  # test 파일명 없으면 final test를 실행차단
        raise ValueError("TEST_FILE_NAME is required for final test.")

    # final test 실행 설정 구성
    # confirm_final_test=True는 실수로 test set을 반복 평가하는 것을 막기 위한 장치
    # overwrite는 OVERWRITE_POLICY['test'] 한 곳에서만 제어
    # 기본 False이면 기존 final test 산출물이 있을 때 재실행이 차단되어 결과 덮어쓰기를 방지
    test_config = ml_test.TestConfig(
        test_path=paths.test_path,
        output_dir=OUT_DIR,
        project_root=PROJECT_ROOT,
        label_col=LABEL_COL,
        confirm_final_test=True,
        sample_rows=None,        # final test는 항상 전체 test split으로만 실행
        allow_nan=False,         # feature에 NaN이 있으면 즉시 에러
        overwrite=OVERWRITE_POLICY["test"],
    )
    test_result = ml_test.test_xgb(test_config)

    print("test_metrics_path:", test_result.metrics_path)
    print("test_metrics:", test_result.test_metrics["summary"])

    return test_result

def main() -> None:
    """실제 데이터 기준으로 입력 확인, 학습, validation, 선택적 final test를 순서대로 실행"""

    # 설정 셀에서 한 번 만든 INPUT_PATHS를 전체 실행 흐름에서 재사용
    # 경로를 main() 안에서 다시 조립하지 않아야 노트북 상단 설정값만 바꾸면 전체 흐름이 같이 바뀜
    paths = INPUT_PATHS
    # 학습 전에 입력 파일 경로, feature 개수, feature hash, 출력 경로를 확인
    preview_inputs(paths)

    # 학습/validation 실행 여부를 설정값으로 제어, False이면 입력 확인만 하고 종료합니다.
    if not RUN_TRAIN_AND_VALIDATION:
        print("Train/validation skipped. Set RUN_TRAIN_AND_VALIDATION=True to train.")
        return

    # train -> validation 순서로 실행
    try:
        run_train_and_validation(paths)
    except ImportError as exc:
        raise SystemExit(str(exc))

    # RUN_FINAL_TEST=True일 때만 최종 test 평가 실행
    # final test는 SAMPLE_ROWS=None인 full run 이후에만 실행
    if RUN_FINAL_TEST:
        run_final_test(paths)
    else:
        print("Final test skipped. Set SAMPLE_ROWS=None and RUN_FINAL_TEST=True only for final evaluation.")



In [ ]:
main()